In [1]:
from datasets import load_dataset

In [2]:
ds = load_dataset("yandex/yambda", data_dir="flat/50m", data_files="likes.parquet")

In [3]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['uid', 'timestamp', 'item_id', 'is_organic'],
        num_rows: 881456
    })
})


In [4]:
dataset=ds['train']
print(dataset.features)

{'uid': Value('uint32'), 'timestamp': Value('uint32'), 'item_id': Value('uint32'), 'is_organic': Value('uint8')}


In [5]:
dataset.to_parquet("likes_50m.parquet")

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

11458928

In [6]:
import polars as pl

In [8]:
# lf=pl.scan_parquet("data/raw/likes_50m.parquet")
lf=pl.scan_parquet("data/raw/likes.parquet")
metrics_plan=(
    lf.select([
        pl.col("uid").n_unique().alias("n_users"),
        pl.col("item_id").n_unique().alias("n_items"),
        pl.len().alias("n_interactions"),
        (pl.col("is_organic")==1).sum().alias("organic_interactions")
    ])
    .with_columns([
        (1.0-(pl.col("n_interactions")/(pl.col("n_users")*pl.col("n_items")))).alias("sparsity"),
        (pl.col("organic_interactions")/pl.col("n_interactions")).alias("organic_ratio")
    ])
)

print("===Базовые метрики датасета ===")
print(metrics_plan.collect())

===Базовые метрики датасета ===
shape: (1, 6)
┌─────────┬─────────┬────────────────┬──────────────────────┬──────────┬───────────────┐
│ n_users ┆ n_items ┆ n_interactions ┆ organic_interactions ┆ sparsity ┆ organic_ratio │
│ ---     ┆ ---     ┆ ---            ┆ ---                  ┆ ---      ┆ ---           │
│ u32     ┆ u32     ┆ u32            ┆ u32                  ┆ f64      ┆ f64           │
╞═════════╪═════════╪════════════════╪══════════════════════╪══════════╪═══════════════╡
│ 8283    ┆ 181304  ┆ 881456         ┆ 502201               ┆ 0.999413 ┆ 0.56974       │
└─────────┴─────────┴────────────────┴──────────────────────┴──────────┴───────────────┘


In [9]:
import numpy as np

In [10]:
item_stats = (
    lf.group_by("item_id")
    .agg(pl.len().alias("interactions"))
    .sort("interactions", descending=True)
    .collect()
)

def calculate_gini(array: np.ndarray) -> float:
    """Вычисляет индекс Джини для массива значений."""
    array = np.sort(array).astype(np.float64)
    index = np.arange(1, array.shape[0] + 1)
    n = array.shape[0]
    return ((np.sum((2 * index - n  - 1) * array)) / (n * np.sum(array)))

# Переводим колонку в numpy-массив для быстрых математических операций
gini_index = calculate_gini(item_stats["interactions"].to_numpy())

print(f"\n=== Анализ смещения ===")
print(f"Индекс Джини: {gini_index:.4f}")
# Если значение > 0.9 — датасет имеет экстремальный "тяжелый хвост".
# Это значит, что модели придется жестко штрафовать за предсказание популярных треков.


=== Анализ смещения ===
Индекс Джини: 0.6833


In [11]:
# Сортируем данные внутри партиций по каждому пользователю
session_plan = (
    lf.sort(["uid", "timestamp"])
    .with_columns([
        # diff() считает разницу с предыдущей строкой, over() изолирует расчеты внутри каждого uid
        pl.col("timestamp").diff().over("uid").alias("time_delta")
    ])
    .filter(pl.col("time_delta").is_not_null()) # Убираем первый лайк пользователя (ему не с чем сравнивать)
    .select([
        pl.col("time_delta").mean().alias("mean_time_between_likes"),
        pl.col("time_delta").median().alias("median_time_between_likes")
    ])
)

print("\n=== Временные характеристики (секунды) ===")
print(session_plan.collect())


=== Временные характеристики (секунды) ===
shape: (1, 2)
┌─────────────────────────┬───────────────────────────┐
│ mean_time_between_likes ┆ median_time_between_likes │
│ ---                     ┆ ---                       │
│ f64                     ┆ f64                       │
╞═════════════════════════╪═══════════════════════════╡
│ 142149.401041           ┆ 500.0                     │
└─────────────────────────┴───────────────────────────┘


TypeError: '>=' not supported between instances of 'str' and 'int'

## Polars: документация к использованным операциям

В ноутбуке применяется **ленивый API Polars**: сначала формируется план вычислений (`LazyFrame`), а фактическое чтение и расчёт выполняются только при вызове `collect()`. Это позволяет оптимизатору читать только нужные столбцы и обрабатывать большие файлы частями.

| Конструкция | Назначение | Документация |
|---|---|---|
| `pl.scan_parquet(path)` | Лениво сканирует parquet без немедленной загрузки всего файла в память | [scan_parquet](https://docs.pola.rs/api/python/stable/reference/api/polars.scan_parquet.html) |
| `select(...)` | Выбирает столбцы и вычисляет выражения | [LazyFrame API](https://docs.pola.rs/api/python/stable/reference/lazyframe/) |
| `with_columns(...)` | Добавляет или заменяет вычисляемые столбцы | [выражения Polars](https://docs.pola.rs/user-guide/expressions/) |
| `filter(...)` | Оставляет строки, удовлетворяющие условию | [LazyFrame.filter](https://docs.pola.rs/api/python/stable/reference/lazyframe/api/polars.LazyFrame.filter.html) |
| `group_by(...).agg(...)` | Группирует строки и считает агрегаты: `sum`, `mean`, `n_unique`, `pl.len()` | [агрегации](https://docs.pola.rs/user-guide/expressions/aggregation/) |
| `diff().over("uid")` | Считает разность с предыдущей строкой отдельно внутри каждого пользователя | [оконные функции](https://docs.pola.rs/user-guide/expressions/window-functions/) |
| `sort(...)` | Сортирует данные перед ранжированием или расчётом временных интервалов | [LazyFrame API](https://docs.pola.rs/api/python/stable/reference/lazyframe/) |
| `collect(engine="streaming")` | Выполняет ленивый план streaming-движком и возвращает `DataFrame` | [LazyFrame.collect](https://docs.pola.rs/api/python/stable/reference/lazyframe/api/polars.LazyFrame.collect.html) |
| `write_parquet(...)` / `sink_parquet(...)` | Сохраняет результат в parquet; `sink_parquet` подходит для потоковой записи больших результатов | [источники и sinks](https://docs.pola.rs/user-guide/lazy/sources_sinks/) |

> Для объёмных расчётов в этом проекте следует использовать `collect(engine="streaming")`. Это актуальный эквивалент старого `collect(streaming=True)`: данные обрабатываются пакетами, что снижает нагрузку на оперативную память.

Полное руководство: [Polars User Guide](https://docs.pola.rs/user-guide/).

## Практика Polars на датасете Yambda

Ниже показан типовой рабочий процесс на нашем parquet-файле:

1. `scan_parquet` создаёт ленивый `LazyFrame` и не загружает весь файл сразу.
2. `select` оставляет нужные поля, `with_columns` создаёт новые признаки, `filter` отбирает строки.
3. `group_by(...).agg(...)` считает метрики на нужном зерне.
4. `sort` и `limit` формируют результат для просмотра.
5. `collect(engine="streaming")` выполняет план пакетами. До этого шага Polars только оптимизирует запрос.
6. `sink_parquet` сохраняет большой ленивый результат напрямую в parquet.

В примере пятисекундный `timestamp` переводится в условный день, после чего рассчитываются DAU, активность и organic ratio.

In [13]:
from pathlib import Path
import polars as pl

SOURCE_PATH = Path("data/raw/likes.parquet")
PROCESSED_DIR = Path("data/processed")
REQUIRED_COLUMNS = ["uid", "timestamp", "item_id", "is_organic"]
DAY_IN_FIVE_SECOND_BINS = 24 * 60 * 60 // 5

# LazyFrame: на этой строке данные ещё не загружаются целиком.
likes_lf = pl.scan_parquet(SOURCE_PATH).select(REQUIRED_COLUMNS)
print(likes_lf.collect_schema())

daily_example = (
    likes_lf
    .with_columns(
        (pl.col("timestamp").cast(pl.UInt64) // DAY_IN_FIVE_SECOND_BINS)
        .cast(pl.UInt32)
        .alias("time_period")
    )
    .group_by("time_period")
    .agg(
        pl.col("uid").n_unique().alias("dau"),
        pl.len().alias("interactions"),
        pl.col("is_organic").mean().alias("organic_ratio"),
    )
    .sort("time_period")
    .limit(5)
    .collect(engine="streaming")
)
print(daily_example.to_dicts())

Schema({'uid': UInt32, 'timestamp': UInt32, 'item_id': UInt32, 'is_organic': UInt8})
[{'time_period': 0, 'dau': 77, 'interactions': 178, 'organic_ratio': 0.4044943820224719}, {'time_period': 1, 'dau': 183, 'interactions': 798, 'organic_ratio': 0.6365914786967418}, {'time_period': 2, 'dau': 199, 'interactions': 575, 'organic_ratio': 0.5182608695652174}, {'time_period': 3, 'dau': 184, 'interactions': 745, 'organic_ratio': 0.6818791946308724}, {'time_period': 4, 'dau': 114, 'interactions': 272, 'organic_ratio': 0.6286764705882353}]


## Анализ аномалий

Проверяются два разных типа отклонений:

- **ошибки качества строк**: пропуски, значения `is_organic` вне `{0, 1}` и полные дубликаты события;
- **поведенческие кандидаты**: пользователи, треки и дни с активностью выше 99-го перцентиля.

Поведенческие кандидаты не удаляются автоматически: популярный трек, активный пользователь или всплеск трафика могут быть корректным бизнес-событием. При очистке удаляются только невалидные строки и повторные полные события.

In [14]:
quality_stats = (
    likes_lf.select(
        pl.len().alias("rows"),
        pl.any_horizontal(pl.all().is_null()).sum().alias("rows_with_nulls"),
        (~pl.col("is_organic").is_in([0, 1])).sum().alias("invalid_is_organic"),
        pl.struct(REQUIRED_COLUMNS).n_unique().alias("unique_rows"),
    )
    .collect(engine="streaming")
    .to_dicts()[0]
)
duplicate_rows = quality_stats["rows"] - quality_stats["unique_rows"]

user_activity = likes_lf.group_by("uid").agg(pl.len().alias("interactions"))
item_activity = likes_lf.group_by("item_id").agg(pl.len().alias("interactions"))
daily_activity = (
    likes_lf
    .with_columns(
        (pl.col("timestamp").cast(pl.UInt64) // DAY_IN_FIVE_SECOND_BINS)
        .alias("time_period")
    )
    .group_by("time_period")
    .agg(pl.len().alias("interactions"))
)

def high_activity_profile(scope: str, plan: pl.LazyFrame) -> dict:
    threshold = (
        plan.select(
            pl.col("interactions").quantile(0.99, interpolation="nearest")
            .alias("p99")
        )
        .collect(engine="streaming")
        .item()
    )
    candidates = (
        plan.select((pl.col("interactions") > threshold).sum().alias("n"))
        .collect(engine="streaming")
        .item()
    )
    return {"scope": scope, "threshold": float(threshold), "candidates": candidates}

user_profile = high_activity_profile("users", user_activity)
item_profile = high_activity_profile("items", item_activity)
day_profile = high_activity_profile("days", daily_activity)

anomaly_summary = pl.DataFrame([
    {"scope": "quality", "metric": "rows_with_nulls", "threshold": 0.0, "candidates": quality_stats["rows_with_nulls"], "interpretation": "Строки с пропусками"},
    {"scope": "quality", "metric": "invalid_is_organic", "threshold": 0.0, "candidates": quality_stats["invalid_is_organic"], "interpretation": "Флаг вне {0, 1}"},
    {"scope": "quality", "metric": "exact_duplicates", "threshold": 0.0, "candidates": duplicate_rows, "interpretation": "Повторные полные события"},
    {"scope": "users", "metric": "interactions_above_p99", "threshold": user_profile["threshold"], "candidates": user_profile["candidates"], "interpretation": "Сверхактивные пользователи"},
    {"scope": "items", "metric": "interactions_above_p99", "threshold": item_profile["threshold"], "candidates": item_profile["candidates"], "interpretation": "Сверхпопулярные треки"},
    {"scope": "days", "metric": "interactions_above_p99", "threshold": day_profile["threshold"], "candidates": day_profile["candidates"], "interpretation": "Дни со всплеском активности"},
])
print(anomaly_summary.to_dicts())

[{'scope': 'quality', 'metric': 'rows_with_nulls', 'threshold': 0.0, 'candidates': 0, 'interpretation': 'Строки с пропусками'}, {'scope': 'quality', 'metric': 'invalid_is_organic', 'threshold': 0.0, 'candidates': 0, 'interpretation': 'Флаг вне {0, 1}'}, {'scope': 'quality', 'metric': 'exact_duplicates', 'threshold': 0.0, 'candidates': 6405, 'interpretation': 'Повторные полные события'}, {'scope': 'users', 'metric': 'interactions_above_p99', 'threshold': 846.0, 'candidates': 83, 'interpretation': 'Сверхактивные пользователи'}, {'scope': 'items', 'metric': 'interactions_above_p99', 'threshold': 61.0, 'candidates': 1806, 'interpretation': 'Сверхпопулярные треки'}, {'scope': 'days', 'metric': 'interactions_above_p99', 'threshold': 1328.0, 'candidates': 15, 'interpretation': 'Дни со всплеском активности'}]


### Результаты проверки

- В 881 456 исходных строках нет пропусков и невалидных значений `is_organic`.
- Обнаружено 6 405 повторных полных событий; после дедупликации остаётся 875 051 строка.
- Выше p99 находятся 83 пользователя (> 846 взаимодействий), 1 806 треков (> 61) и 15 условных дней (> 1 328).
- Эти p99-наблюдения сохранены в данных: они описывают heavy-tail и всплески активности, а не доказанные ошибки.

Следующая ячейка сохраняет очищенный датасет и машинно-читаемую сводку аномалий.

In [15]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DATASET_PATH = PROCESSED_DIR / "likes_clean.parquet"
ANOMALY_SUMMARY_PATH = PROCESSED_DIR / "anomaly_summary.parquet"

valid_row = (
    ~pl.any_horizontal(pl.all().is_null())
    & pl.col("is_organic").is_in([0, 1])
)
clean_lf = (
    likes_lf
    .filter(valid_row)
    .unique(subset=REQUIRED_COLUMNS, keep="first", maintain_order=False)
    .sort(REQUIRED_COLUMNS)
)
clean_lf.sink_parquet(CLEAN_DATASET_PATH, compression="zstd", mkdir=True)
anomaly_summary.write_parquet(ANOMALY_SUMMARY_PATH, compression="zstd")

saved_rows = (
    pl.scan_parquet(CLEAN_DATASET_PATH)
    .select(pl.len().alias("rows"))
    .collect(engine="streaming")
    .item()
)
print(f"Saved {CLEAN_DATASET_PATH}: {saved_rows:,} rows; removed {quality_stats['rows'] - saved_rows:,} quality anomalies")
print(f"Saved {ANOMALY_SUMMARY_PATH}: {anomaly_summary.height} checks")

Saved data\processed\likes_clean.parquet: 875,051 rows; removed 6,405 quality anomalies
Saved data\processed\anomaly_summary.parquet: 6 checks
